# Transfer Learning — Reusing Pretrained Models

## Goal
Instead of training from scratch, reuse a model already trained on millions of images.
Fine-tune only the last layer for our specific task.

## Why Transfer Learning?
Training ResNet from scratch on ImageNet takes weeks on GPUs.
Fine-tuning takes minutes on a laptop.

## What We Build
Use ResNet-18 pretrained on ImageNet.
Replace the final layer for a new classification task.
Freeze all layers except the last one — train only the classifier.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
import numpy as np
import matplotlib.pyplot as plt

## 1. Load Dataset with ImageNet Preprocessing
ResNet expects 224x224 images normalized with ImageNet statistics.

In [3]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

# ImageNet normalization — required for pretrained models
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),  # MNIST is grayscale, ResNet needs 3 channels
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

# Use a subset of MNIST for speed
full_dataset = datasets.MNIST(
    root="../data/raw",
    train=True,
    download=True,
    transform=transform
)

# Use only 2000 samples — transfer learning needs less data
subset_size = 2000
subset, _ = random_split(full_dataset, [subset_size, len(full_dataset) - subset_size])

train_size = int(0.8 * subset_size)
val_size = subset_size - train_size
train_dataset, val_dataset = random_split(subset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

Training samples: 1600
Validation samples: 400
